# Translating Wikipedia through LLMs
### Workshop · Arianit Dobroshi

Hands-on notebook: every cell can be run or copied as is.
Steps go from simplest to most complex, from what exists to what still
needs building.

| Step | What you do | Status |
|---|---|---|
| 0 | Why: the numbers | |
| 1 | What a Wikipedia page is | ✅ |
| 2 | Build a frwiki → sqwiki skill | 🔧 built live |
| 3 | Run the skill by hand | ✅ |
| 4 | Verification by hand (Wikidata, templates) | ✅ |
| 5 | Improve the skill and commit | 🔧 |
| 6 | The automated harness | ✅ |
| 7 | Choosing models | ✅ / 🔧 |
| 8 | Sharing work through the queue | ✅ |
| 9 | Not built yet, and how to help | 🧭 |

✅ built today · 🔧 partial or built during the session · 🧭 not built yet

*Present as slides:* `jupyter nbconvert workshop-walkthrough.ipynb --to slides --post serve`

## Tabs to open before starting

**The project**
- Skill: https://github.com/arianit/enwiki-sqwiki-translation
- Harness: https://github.com/arianit/wiki-translate-harness
- Architecture diagram: https://github.com/arianit/wiki-translate-harness/blob/main/docs/architecture.svg
- Changelog: https://github.com/arianit/wiki-translate-harness/blob/main/CHANGELOG.md
- Config reference: https://github.com/arianit/wiki-translate-harness/blob/main/config.example.yaml
- Benchmark (issue #1): https://github.com/arianit/wiki-translate-harness/issues/1
- Queue: https://github.com/arianit/wiki-translate-queue/blob/main/totranslate.txt
- Language-tech plan: https://github.com/arianit/albanian-language-tech-plan

**Wikipedia and Wikidata**
- Wikipedias by size: https://meta.wikimedia.org/wiki/List_of_Wikipedias
- sqwiki statistics: https://sq.wikipedia.org/wiki/Special:Statistics
- Articles every Wikipedia should have: https://meta.wikimedia.org/wiki/List_of_articles_every_Wikipedia_should_have
- Demo article (rendered): https://fr.wikipedia.org/wiki/Prizren
- Demo article (source): https://fr.wikipedia.org/w/index.php?title=Prizren&action=edit
- Its Wikidata item: https://www.wikidata.org/wiki/Special:ItemByTitle/frwiki/Prizren
- Wikidata API sandbox: https://www.wikidata.org/wiki/Special:ApiSandbox
- frwiki `{{Lien web}}`: https://fr.wikipedia.org/wiki/Mod%C3%A8le:Lien_web
- sqwiki `{{Cite web}}`: https://sq.wikipedia.org/wiki/Stampa:Cite_web
- Content Translation on sqwiki (log in first): https://sq.wikipedia.org/wiki/Special:ContentTranslation

**Policy**
- WP:LLM-assisted translation: https://en.wikipedia.org/wiki/Wikipedia:LLM-assisted_translation
- WP:Writing articles with LLMs: https://en.wikipedia.org/wiki/Wikipedia:Writing_articles_with_large_language_models
- Content Translation docs: https://www.mediawiki.org/wiki/Content_translation
- Wikimedia User-Agent policy: https://foundation.wikimedia.org/wiki/Policy:User-Agent_policy

**Tools and compute**
- Claude Code skills docs: https://docs.claude.com/en/docs/claude-code/skills
- OpenRouter models: https://openrouter.ai/models
- Experiential: https://github.com/experientiallabs/experiential
- OpenCode: https://github.com/sst/opencode
- Kosovo HPC & AI Competence Centre: https://hpc.kren-ks.eu/

## 0. Why

- Wikipedia: 65M+ articles in 340+ languages. Albanian Wikipedia: ~100k.
- English and several other editions are deep and well sourced. For
  Albanian the content exists; it has not been moved over.
- Goal: 1,000, then 10,000, of the
  [Articles every Wikipedia should have](https://meta.wikimedia.org/wiki/List_of_articles_every_Wikipedia_should_have)
  in Albanian.
- LLMs make translation cheap. The bottleneck moves to **verification,
  review and coordination**.

🔗 **Open now:** [Wikipedias by size](https://meta.wikimedia.org/wiki/List_of_Wikipedias) · [sqwiki statistics](https://sq.wikipedia.org/wiki/Special:Statistics) · [Articles every Wikipedia should have](https://meta.wikimedia.org/wiki/List_of_articles_every_Wikipedia_should_have)

### 🗣 Discussion

- Is 1,000 realistic in a year? Is 10,000? What limits it: compute, money, or reviewers?
- Which articles first: the 'every Wikipedia should have' list, Kosovo/Albania topics, or what readers search for and don't find?
- Who reviews? A translated article nobody reads before publishing is worse than no article.

## Setup

Set everything once here; later cells (Python and `%%bash`) read these.

Wikimedia asks every automated client to identify itself
([User-Agent policy](https://foundation.wikimedia.org/wiki/Policy:User-Agent_policy)).
Needs Python 3.10+, `requests`, `git`; for the run steps also
[Claude Code](https://docs.claude.com/en/docs/claude-code/overview) (`claude`).

Pick a **short** frwiki stub for `DEMO_FR` (a few paragraphs, two or three
citations) so the live run finishes in a few minutes. `Prizren` is used
for looking at wikitext only.

In [ ]:
import os
import re
import requests
from pathlib import Path

settings = {
    "WIKIMEDIA_CONTACT": "your-name (you@example.com)",   # <- change
    "DEMO_FR": "Prizren",           # <- change to a short frwiki stub before the live run
    "DEMO_EN": "Enji (deity)",      # short enwiki article, used in the benchmark
    "WORK": str(Path.home() / "code"),   # where repos get cloned
    "ALLOW_QUEUE_WRITES": "no",     # "yes" only if you have write access to the shared queue
}
os.environ.update(settings)
os.environ["UA"] = f"wiki-translate-workshop/0.1 ({settings['WIKIMEDIA_CONTACT']})"
os.environ["DEMO_SLUG"] = settings["DEMO_FR"].replace(" ", "_")
Path(settings["WORK"]).mkdir(parents=True, exist_ok=True)
Path("out").mkdir(exist_ok=True)

UA = os.environ["UA"]
for k in ["UA", "DEMO_FR", "DEMO_EN", "WORK"]:
    print(f"{k:5} = {os.environ[k]}")

In [ ]:
%%bash
# What is installed?
for c in git python3 claude opencode gh; do
  printf "%-8s " "$c"; command -v "$c" >/dev/null && echo "ok" || echo "missing"
done

## 1. What a Wikipedia page is ✅

Every article is stored as **wikitext**: prose mixed with links,
templates, citations and categories. Fetch one:

In [ ]:
def raw_wikitext(title: str, lang: str = "fr") -> str:
    r = requests.get(
        f"https://{lang}.wikipedia.org/w/index.php",
        params={"title": title, "action": "raw"},
        headers={"User-Agent": UA},
        timeout=30,
    )
    r.raise_for_status()
    return r.text

text = raw_wikitext("Prizren")
print(text[:2000])

🔗 **Open now:** [Rendered page](https://fr.wikipedia.org/wiki/Prizren) · [Edit source](https://fr.wikipedia.org/w/index.php?title=Prizren&action=edit) · [Raw wikitext](https://fr.wikipedia.org/w/index.php?title=Prizren&action=raw)

In [ ]:
links = re.findall(r"\[\[([^\]|#]+)", text)
templates = re.findall(r"\{\{\s*([^|}\n]+)", text)
refs = re.findall(r"<ref[ >]", text)
cats = [l for l in links if l.startswith(("Catégorie:", "Category:"))]

print(f"wikilinks:  {len(links) - len(cats)}")
print(f"templates:  {len(templates)}  e.g. {sorted(set(t.strip() for t in templates))[:10]}")
print(f"references: {len(refs)}")
print(f"categories: {len(cats)}")

### Anatomy

```
== Histoire ==                                   <- section heading
La ville fut fondée au {{s-|XIV}} et devint      <- prose + century template
un centre commercial important. Selon
l'historien [[Jean Dupont]], elle comptait        <- wikilink
environ {{formatnum:15000}} habitants en 1850     <- number template
<ref>{{Ouvrage|nom1=Dupont|titre=...}}</ref>.      <- citation

{{Infobox Commune de France                        <- infobox = template
| nom        = Exemple-sur-Loire                   <-   with named parameters
| population = 12 458
}}

[[Catégorie:Ville de France]]                      <- category
```

| Part | What happens to it |
|---|---|
| Prose | **translated** |
| Wikilinks | **verified**: does an sqwiki article exist, under which title? |
| Templates, infoboxes | **matched or substituted**: does sqwiki have it, with which parameters? |
| Citations, categories | **converted** to sqwiki conventions |

One article, four different jobs. Only one of them is translation.

### 🗣 Discussion

- Which of the four is hardest to get right? Which fails silently (renders fine, but wrong)?
- Many frwiki infoboxes pull their data from Wikidata. Should sqwiki infoboxes do the same, so nothing in them needs translating?
- Red links: leave them, use {{ill}}, or unlink? What does the sqwiki community prefer?

## 2. Build a skill: frwiki → sqwiki 🔧

A **skill** is a folder with one required file, `SKILL.md`:

- **frontmatter** (`name`, `description`): tells the assistant *when* to use it
- **body**: tells it *how*
- optional `references/`: longer material, read only when needed

```
frwiki-sqwiki-translation/
  SKILL.md
  references/
    fr-citation-templates.md
    sqwiki-verified.md
```

This skill covers the **French source side**. For Albanian output rules it
defers to the existing
[`enwiki-sqwiki-translation`](https://github.com/arianit/enwiki-sqwiki-translation)
skill. It is a starting point: part of the exercise is running it and
fixing what breaks.

In [ ]:
%%bash
mkdir -p frwiki-sqwiki-translation/references
ls -la frwiki-sqwiki-translation

### 2.1 `SKILL.md`

In [ ]:
%%writefile frwiki-sqwiki-translation/SKILL.md
---
name: frwiki-sqwiki-translation
description: Translate French Wikipedia (fr.wikipedia.org) articles into Albanian as paste-ready wikitext for sq.wikipedia. Use when the user pastes French wikitext or a fr.wikipedia.org URL and asks for it in Albanian, or mentions frwiki together with sqwiki / Albanian Wikipedia.
---

# frwiki → sqwiki translation

Produce a `.wiki` file an experienced sqwiki editor can paste with minimal
cleanup, plus a short report of everything you could not verify.

## Division of work

- This skill covers the **French source side**: reading French citation
  templates, French dates and numbers, French section headings, and
  frwiki-specific templates.
- For **Albanian output conventions** (title translation, dual
  transliteration, Albanian number formatting, `{{Përkthyer nga}}`
  attribution, safe template substitutions), follow the
  `enwiki-sqwiki-translation` skill if it is available. Its rules apply
  unchanged; only the source language differs.
- If it is not available, apply the minimal output rules at the end of this
  file.

## Workflow

1. **Get the source.** If you have shell access, fetch raw wikitext:
   `curl -s "https://fr.wikipedia.org/w/index.php?title=TITLE&action=raw"`.
   Otherwise ask the user to paste it. Record the revision (oldid) for
   attribution.
2. **Inventory before translating.** List every wikilink, template and
   category in the source. These are verified, not translated.
3. **Verify targets.** For each linked article and each template, check
   whether an sqwiki equivalent exists (Wikidata sitelinks, see
   Verification). Check `references/sqwiki-verified.md` first.
4. **Translate prose.** Meaning over word order. Keep every fact and every
   citation attached to the same sentence it was attached to in French.
5. **Convert citations** using `references/fr-citation-templates.md`.
6. **Convert dates and numbers** (rules below).
7. **Deliver** the `.wiki` file and the report.

## French source conventions to recognise

- **Citation templates**: `{{Ouvrage}}`, `{{Article}}`, `{{Lien web}}`,
  `{{Chapitre}}`, and short forms `{{Harvsp}}` / `{{sfn}}`. Parameter names
  are French. Map them to CS1 templates on sqwiki (see reference file).
  Every converted citation gets `|language=fr` unless the cited work itself
  is in another language (check its title and the `langue=` parameter).
- **Dates**: `1er janvier 2020`, `12 mars 1998`, `{{date|12|mars|1998}}`,
  `{{date-|...}}`. In citations, output ISO: `2020-01-01`.
- **Centuries**: `{{s-|XIV}}` = 14th century, `{{sp-|XIV|e|XV}}` = 14th to
  15th century. Render as Albanian prose ("shekulli XIV") rather than
  keeping the French template.
- **Numbers**: French uses a space (often non-breaking) for thousands and a
  comma for decimals: `12 458`, `3,5`. Also `{{nombre|12458}}`,
  `{{unité|3.5|km}}`, `{{formatnum:12458}}`. Output Albanian format:
  `12.458`, `3,5`.
- **Infoboxes**: many frwiki infoboxes (for example `{{Infobox Biographie2}}`)
  pull most of their content from Wikidata and have few or no parameters in
  the wikitext. Do not invent parameters. Check which infobox exists on
  sqwiki and which parameter names it really accepts.
- **Maintenance and navigation templates** (`{{Portail|...}}`,
  `{{Palette|...}}`, `{{Article de qualité}}`, `{{Ébauche|...}}`): drop
  them. They have no sqwiki meaning.
- **Section headings**: translate; do not keep French headings. Common ones:
  Histoire, Géographie, Démographie, Biographie, Œuvres, Notes et
  références, Voir aussi, Bibliographie, Liens externes.
- **Interlanguage links** `{{Lien|fr=...|lang=en|trad=...}}`: treat the
  foreign title as the concept; check for an sqwiki article first.

## Verification

Batch-check up to 50 titles per request against Wikidata:

```
https://www.wikidata.org/w/api.php?action=wbgetentities&sites=frwiki&titles=A|B|C&props=sitelinks&sitefilter=frwiki|sqwiki&format=json
```

- sqwiki sitelink present → use that exact title.
- No sitelink → use `{{ill|Albanian display|fr|French title}}` for notable
  topics, plain text for generic words.

Check template existence on sqwiki (template namespace is `Stampa`):

```
https://sq.wikipedia.org/w/api.php?action=query&titles=Stampa:Cite_book|Stampa:Cite_web&format=json
```

Without shell access, do these checks one at a time with web search and
say so in the report.

## Minimal output rules (only if enwiki-sqwiki-translation is absent)

- CS1 parameter names stay in English (`|title=`, never `|titulli=`).
- Albanian numbers: `.` for thousands, `,` for decimals.
- ISO dates in citations.
- Slavic and other non-Latin names: Albanian form, original in
  parentheses on first mention.
- Categories use the `Kategoria:` prefix and must exist on sqwiki.
- Talk-page attribution: `{{Përkthyer nga|fr|French title|oldid}}`
  (confirm exact parameters on sqwiki).

## Report (always deliver)

- Links: confirmed / not found.
- Templates: kept / substituted / dropped, with reason.
- Citations converted, and any where the language was a guess.
- Anything you did not verify, stated plainly.

### 2.2 French citation templates

The parameter lists cover common cases. Step 4.4 shows how to check them
against the live templates.

In [ ]:
%%writefile frwiki-sqwiki-translation/references/fr-citation-templates.md
# French citation templates → sqwiki CS1

Add `|language=fr` to every converted citation unless the cited work is in
another language.

## {{Ouvrage}} → {{cite book}}

| frwiki | sqwiki CS1 |
|---|---|
| prénom1 / nom1 (or auteur1) | first1 / last1 (or author1) |
| titre | title |
| sous-titre | append to title after a colon |
| éditeur | publisher |
| lieu | location |
| année / date | year / date (ISO) |
| isbn | isbn |
| pages totales | omit |
| passage | page / pages |
| lire en ligne | url |
| consulté le | access-date (ISO) |
| langue | language (ISO code) |

## {{Article}} → {{cite journal}} (or {{cite news}} for newspapers)

| frwiki | sqwiki CS1 |
|---|---|
| prénom1 / nom1 | first1 / last1 |
| titre | title |
| périodique | journal (or work / newspaper) |
| date | date (ISO) |
| volume | volume |
| numéro | issue |
| pages | pages |
| doi | doi |
| lire en ligne | url |
| consulté le | access-date (ISO) |

## {{Lien web}} → {{cite web}}

| frwiki | sqwiki CS1 |
|---|---|
| auteur / prénom / nom | author / first / last |
| titre | title |
| url | url |
| site | website |
| éditeur | publisher |
| date | date (ISO) |
| consulté le | access-date (ISO) |
| langue | language |

## {{Chapitre}} → {{cite book}} with |chapter=

| frwiki | sqwiki CS1 |
|---|---|
| titre chapitre | chapter |
| titre ouvrage | title |
| (other parameters) | as {{Ouvrage}} |

### 2.3 Confirmed targets

Starts almost empty. Every target you confirm goes here with how you
confirmed it, so the next run does not repeat the lookup.

In [ ]:
%%writefile frwiki-sqwiki-translation/references/sqwiki-verified.md
# Confirmed sqwiki targets

| frwiki | sqwiki | how confirmed |
|---|---|---|

### 2.4 Install

**Claude Code:** copy into `~/.claude/skills/`, start a new session, run
`/skills` to check it is listed.

**Claude on the web:** upload the zip as a skill in your Claude settings
(skills must be enabled for your account).

In [ ]:
%%bash
mkdir -p ~/.claude/skills
cp -r frwiki-sqwiki-translation ~/.claude/skills/
zip -qr frwiki-sqwiki-translation.zip frwiki-sqwiki-translation
ls ~/.claude/skills/ frwiki-sqwiki-translation.zip

### 2.5 Install the existing Albanian-output skills

The new skill defers to `enwiki-sqwiki-translation` for Albanian output
rules. Clone it and link every skill folder in it into `~/.claude/skills/`.
The harness's default config also expects `wikiterms` and `wikiqa` there;
if the public repo doesn't contain them yet, the config cell in 6.2 drops
them and warns.

In [ ]:
%%bash
cd "$WORK"
[ -d enwiki-sqwiki-translation ] || git clone -q https://github.com/arianit/enwiki-sqwiki-translation
cd enwiki-sqwiki-translation && git pull -q
mkdir -p ~/.claude/skills
find . -name SKILL.md -not -path "./.git/*" | while read f; do
  d=$(cd "$(dirname "$f")" && pwd)
  ln -sfn "$d" ~/.claude/skills/"$(basename "$d")"
  echo "linked $(basename "$d")"
done
ls -l ~/.claude/skills/

🔗 **Open now:** [Skill repo](https://github.com/arianit/enwiki-sqwiki-translation) · [Claude Code skills docs](https://docs.claude.com/en/docs/claude-code/skills)

### 2.6 First test

**Interactive** (best for the live demo, the room sees each step):

```bash
claude
```
then type `/skills` to confirm both skills are listed, and paste:

```text
Translate this French Wikipedia article into Albanian for sq.wikipedia:
https://fr.wikipedia.org/wiki/<DEMO_FR>
Give me the .wiki output and the verification report.
```

**Non-interactive** (the cell below): same request with `claude -p`,
output saved to `out/`. `--allowedTools` lets it call `curl` and write
files without stopping to ask. If it still stops on a permission, add the
tool it names to the list.

Check: did the skill trigger? Citations as CS1 with `|language=fr`?
`{{Portail}}` dropped? Numbers converted to `12.458`?

In [ ]:
%%bash
claude -p "Use the frwiki-sqwiki-translation skill. Translate https://fr.wikipedia.org/wiki/$DEMO_SLUG into Albanian for sq.wikipedia. Write the wikitext to out/$DEMO_SLUG.v1.wiki and the verification report to out/$DEMO_SLUG.v1.report.md. Then print the report." \
  --allowedTools "Bash(curl:*)" "Read" "Write"

### 🗣 Discussion

- What belongs in a skill (judgment, conventions) and what belongs in code (lookups, fixes that must always happen)?
- One skill per source language, or one generic skill with a section per language?
- Who maintains the terminology? Linguists editing a markdown file in git: realistic, or does it need a friendlier interface?
- Does the base skill already handle French well without this? Check live before assuming.

## 3. Run the skill by hand ✅

**Claude on the web** (claude.ai, skill uploaded as zip): paste this with
the raw wikitext.

```text
Use the frwiki-sqwiki-translation skill on the article below.
Work section by section. Verify every wikilink and template before using it.
At the end give me: (1) the full .wiki file, (2) the verification report.

<paste raw wikitext here>
```

**Claude Code**: the cells below run the existing English skill and the
new French one, so the room can compare. Same model, same machine; only
the skill and the source differ.

In [ ]:
%%bash
# Existing skill, English source
EN_SLUG=$(echo "$DEMO_EN" | tr ' ' '_')
claude -p "Use the enwiki-sqwiki-translation skill. Translate https://en.wikipedia.org/wiki/$EN_SLUG into Albanian for sq.wikipedia. Write the wikitext to out/$EN_SLUG.wiki and the report to out/$EN_SLUG.report.md. Then print the report." \
  --allowedTools "Bash(curl:*)" "Read" "Write"

In [ ]:
%%bash
# Same article translated from a pasted file instead of a URL
# (what you'd do on the web, or offline)
curl -s -A "$UA" "https://fr.wikipedia.org/w/index.php?title=$DEMO_SLUG&action=raw" > "out/$DEMO_SLUG.fr.wiki"
wc -l "out/$DEMO_SLUG.fr.wiki"
claude -p "Use the frwiki-sqwiki-translation skill on the French wikitext in out/$DEMO_SLUG.fr.wiki. Do not fetch anything; work only from the file and your own knowledge, and mark every link and template you could not verify. Write out/$DEMO_SLUG.offline.wiki and out/$DEMO_SLUG.offline.report.md." \
  --allowedTools "Read" "Write"

Compare the report from the online run (2.6) with the offline one: the
offline run is roughly what Claude on the web can do without shell access.

In [ ]:
%%bash
ls -la out/
echo "--- unverified items, online vs offline ---"
grep -ci "not verified\|unverified\|not found" out/*.report.md 2>/dev/null || echo "(no reports yet)"

### Claude Code vs. Claude on the web

| | Claude Code (command line) | Claude on the web |
|---|---|---|
| Fetch source | `curl ... action=raw` | paste, or single page fetch |
| Link checks | 50 titles per Wikidata call, exact JSON | one search or page fetch at a time |
| Template parameters | reads live template source | approximate, from rendered pages |
| Citation language from URL | can visit each URL | rarely practical |
| Cache between sessions | files on disk | none |
| Long articles | can split and loop | manual copy per section |

Same skill, same rules. The web can browse, but coarsely: a weaker mode of
the same skill, not an equivalent.

### Compared with Wikimedia's Content Translation

| | Content Translation (CX) | This pipeline |
|---|---|---|
| Draft from | machine translation (Google and others) | LLM following a versioned rule set |
| Links, categories | carried over automatically | verified via Wikidata, reported |
| Templates | carried over where mapped | checked on the target wiki, substituted or dropped |
| Human role | writes the translation from the MT draft | reviews the finished draft |
| Model choice, benchmarks | none | yes |

Policy is moving: English Wikipedia banned LLM-generated article text in
March 2026, with a narrow exception for translation under
[WP:LLM-assisted translation](https://en.wikipedia.org/wiki/Wikipedia:LLM-assisted_translation)
(translator must know both languages, check sources, remove errors).

🔗 **Open now:** [Content Translation on sqwiki](https://sq.wikipedia.org/wiki/Special:ContentTranslation) · [CX docs](https://www.mediawiki.org/wiki/Content_translation) · [WP:LLM-assisted translation](https://en.wikipedia.org/wiki/Wikipedia:LLM-assisted_translation) · [WP:Writing articles with LLMs](https://en.wikipedia.org/wiki/Wikipedia:Writing_articles_with_large_language_models)

### 🗣 Discussion

- Is Claude on the web good enough for an occasional editor translating one article a month?
- CX puts the human in the writing seat; this puts them in the reviewing seat. Which produces better articles? Which will the sqwiki community accept?
- What should sqwiki's own rule on LLM-assisted translation be? Is there one today?
- Attribution: the source oldid in the edit summary and {{Përkthyer nga}} on the talk page. Enough, or should LLM use also be declared?

## 4. Verification by hand ✅

What the skill asks the model to do, and what the harness automates later.

### 4.1 Which linked articles exist on sqwiki (50 per call)

In [ ]:
def sq_sitelinks(titles: list[str], source: str = "frwiki") -> dict:
    """Map each source title to its sqwiki title, or None."""
    out = {}
    for i in range(0, len(titles), 50):
        batch = titles[i : i + 50]
        r = requests.get(
            "https://www.wikidata.org/w/api.php",
            params={
                "action": "wbgetentities",
                "sites": source,
                "titles": "|".join(batch),
                "props": "sitelinks",
                "sitefilter": f"{source}|sqwiki",
                "format": "json",
            },
            headers={"User-Agent": UA},
            timeout=30,
        )
        r.raise_for_status()
        for ent in r.json().get("entities", {}).values():
            src = ent.get("sitelinks", {}).get(source, {}).get("title")
            sq = ent.get("sitelinks", {}).get("sqwiki", {}).get("title")
            if src:
                out[src] = sq
        for t in batch:
            out.setdefault(t, None)
    return out

article_links = sorted({l.strip() for l in links if ":" not in l})
result = sq_sitelinks(article_links)
for fr, sq in result.items():
    print(f"{'OK ' if sq else '-- '} {fr:40} -> {sq or '(no sqwiki article)'}")

### 4.2 Notable enough for an `{{ill}}` link?

Rough proxy used by the harness: how many Wikipedias have an article on it.

In [ ]:
def sitelink_count(title: str, source: str = "frwiki") -> int:
    r = requests.get(
        "https://www.wikidata.org/w/api.php",
        params={"action": "wbgetentities", "sites": source, "titles": title,
                "props": "sitelinks", "format": "json"},
        headers={"User-Agent": UA}, timeout=30,
    )
    ents = r.json().get("entities", {})
    return max((len(e.get("sitelinks", {})) for e in ents.values()), default=0)

print(sitelink_count("Ligue de Prizren"))

### 4.3 Which templates exist on sqwiki (namespace `Stampa`)

In [ ]:
def sq_templates_exist(names: list[str]) -> dict:
    titles = [f"Stampa:{n}" for n in names]
    r = requests.get(
        "https://sq.wikipedia.org/w/api.php",
        params={"action": "query", "titles": "|".join(titles), "format": "json"},
        headers={"User-Agent": UA}, timeout=30,
    )
    pages = r.json()["query"]["pages"].values()
    return {p["title"]: "missing" not in p for p in pages}

print(sq_templates_exist(["Cite book", "Cite web", "Cite journal", "Ill"]))

### 4.4 Which parameters a template really accepts

Uses TemplateData where it exists. Empty list: read the template's
documentation page instead. Use this to correct the table from step 2.2.

In [ ]:
def template_params(template: str, lang: str = "fr", ns: str = "Modèle") -> list:
    r = requests.get(
        f"https://{lang}.wikipedia.org/w/api.php",
        params={"action": "templatedata", "titles": f"{ns}:{template}", "format": "json"},
        headers={"User-Agent": UA}, timeout=30,
    )
    for p in r.json().get("pages", {}).values():
        return sorted(p.get("params", {}).keys())
    return []

print("frwiki {{Lien web}}:", template_params("Lien web"))
print("sqwiki {{Cite web}}:", template_params("Cite web", lang="sq", ns="Stampa"))

🔗 **Open now:** [Wikidata item for the demo article](https://www.wikidata.org/wiki/Special:ItemByTitle/frwiki/Prizren) · [Wikidata API sandbox](https://www.wikidata.org/wiki/Special:ApiSandbox) · [Same query as a URL](https://www.wikidata.org/w/api.php?action=wbgetentities&sites=frwiki&titles=Prizren&props=sitelinks&sitefilter=frwiki|sqwiki&format=json) · [frwiki {{Lien web}}](https://fr.wikipedia.org/wiki/Mod%C3%A8le:Lien_web) · [sqwiki {{Cite web}}](https://sq.wikipedia.org/wiki/Stampa:Cite_web)

### 🗣 Discussion

- Wikidata sitelinks as the source of truth for links: where does that break? (Disambiguation pages, articles merged differently on each wiki.)
- Is 'number of Wikipedias with an article' a fair test of notability for an {{ill}} link? What would be better?
- sqwiki infoboxes that kept English parameter names: a problem to fix, or a feature that makes translation easier?

## 5. Improve the skill and commit 🔧

Fix what went wrong, run the same article again, compare.

In [ ]:
%%bash
cd frwiki-sqwiki-translation
git init -q 2>/dev/null
git add .
git -c user.name="workshop" -c user.email="workshop@example.com" commit -qm "frwiki-sqwiki-translation: first draft" && git log --oneline

Edit `SKILL.md` or the reference files (live, with the room), then run the
cells below: reinstall, translate again as v2, compare with v1 from 2.6.

In [ ]:
%%bash
cd frwiki-sqwiki-translation && git diff --stat && git diff | head -60

In [ ]:
%%bash
cp -r frwiki-sqwiki-translation ~/.claude/skills/
claude -p "Use the frwiki-sqwiki-translation skill. Translate https://fr.wikipedia.org/wiki/$DEMO_SLUG into Albanian for sq.wikipedia. Write the wikitext to out/$DEMO_SLUG.v2.wiki and the verification report to out/$DEMO_SLUG.v2.report.md." \
  --allowedTools "Bash(curl:*)" "Read" "Write"
diff -u "out/$DEMO_SLUG.v1.wiki" "out/$DEMO_SLUG.v2.wiki" | head -80

In [ ]:
%%bash
cd frwiki-sqwiki-translation
git -c user.name="workshop" -c user.email="workshop@example.com" commit -qam "Improve French source handling (workshop)" || echo "(nothing changed yet)"
git log --oneline

Send it upstream:

```bash
gh repo fork arianit/enwiki-sqwiki-translation --clone
gh issue create --repo arianit/enwiki-sqwiki-translation \
  --title "French source support: citation template mapping" \
  --body "Tested on fr:<article>. Findings: ..."
```

### 🗣 Discussion

- How do we review a change to a prompt? A code change has tests; what's the equivalent for a skill?
- Should every skill change be re-run against a fixed set of articles before merging?

## 6. The automated harness ✅

[`wiki-translate-harness`](https://github.com/arianit/wiki-translate-harness)
does steps 3 and 4 for you, per chunk: fetch, split, verify, translate,
validate, repair, cache, report. It holds **no translation rules**; it
loads the skill.

```
fetch → split → cache? → prompt (skill + verified facts) → engine
      → validate → repair or flag for review → assemble → post-process → .wiki + report
```

Engines: Claude Code CLI (default, no API key), OpenRouter, any local
OpenAI-compatible server, Experiential, OpenCode Go.

🔗 **Open now:** [Harness repo](https://github.com/arianit/wiki-translate-harness) · [Architecture diagram](https://github.com/arianit/wiki-translate-harness/blob/main/docs/architecture.svg) · [config.example.yaml](https://github.com/arianit/wiki-translate-harness/blob/main/config.example.yaml)

### 6.1 Install

Clones into `$WORK`, makes a virtual environment, installs the harness.
Takes a minute or two: run it before the session.

In [ ]:
%%bash
cd "$WORK"
[ -d wiki-translate-harness ] || git clone -q https://github.com/arianit/wiki-translate-harness
cd wiki-translate-harness && git pull -q
[ -d .venv ] || python3 -m venv .venv
.venv/bin/pip install -q -e .
.venv/bin/wiki-translation-harness --help | head -40

### 6.2 Config files for the workshop

Three configs from `config.example.yaml`:

- `config.workshop.yaml`: default engine (Claude Code CLI), output to
  `./output` instead of the shared queue repo
- `config.fr.yaml`: same, plus today's `frwiki-sqwiki-translation` skill
- `config.openrouter.yaml`: OpenRouter engine, needed for benchmark mode

In [ ]:
H = Path(os.environ["WORK"]) / "wiki-translate-harness"
base = (H / "config.example.yaml").read_text()

def setkey(text, key, value):
    new, n = re.subn(rf"^{key}:.*$", f"{key}: {value}", text, flags=re.M)
    assert n == 1, f"key {key!r} not found once in config.example.yaml"
    return new

def skill_block(names):
    found = [n for n in names if (Path.home() / ".claude/skills" / n / "SKILL.md").exists()]
    for n in set(names) - set(found):
        print(f"warning: ~/.claude/skills/{n} not found, left out of skill_path")
    return "skill_path:\n" + "".join(f"  - ~/.claude/skills/{n}\n" for n in found)

def set_skills(text, names):
    new, n = re.subn(r"^skill_path:\n(?:  - .*\n)+", skill_block(names), text, flags=re.M)
    assert n == 1, "skill_path block not found"
    return new

qa = Path.home() / ".claude/skills/wikiqa/SKILL.md"
if not qa.exists():
    print("warning: ~/.claude/skills/wikiqa not found; repair calls may fail. "
          "Point qa_skill_path at your QA skill in the config files below.")

base = setkey(base, "wikimedia_contact", f'"{os.environ["WIKIMEDIA_CONTACT"]}"')
base = setkey(base, "output_dir", "output")
base = set_skills(base, ["enwiki-sqwiki-translation", "wikiterms"])
(H / "config.workshop.yaml").write_text(base)

fr = set_skills(base, ["frwiki-sqwiki-translation", "enwiki-sqwiki-translation", "wikiterms"])
(H / "config.fr.yaml").write_text(fr)

(H / "config.openrouter.yaml").write_text(setkey(base, "provider", "openrouter"))

for f in ["config.workshop.yaml", "config.fr.yaml", "config.openrouter.yaml"]:
    print("wrote", H / f)

### 6.3 Translate one article

Default engine: Claude Code CLI (`claude -p` under your login, no API key).
Watch the log in a second terminal:
`tail -f $WORK/wiki-translate-harness/logs/run.log`

In [ ]:
%%bash
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
wiki-translation-harness --config config.workshop.yaml --title "$DEMO_EN"

In [ ]:
%%bash
# French source, with today's skill added to the prompt
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
wiki-translation-harness --config config.fr.yaml --title "fr:$DEMO_FR"

In [ ]:
%%bash
# What came out
cd "$WORK/wiki-translate-harness"
if ls output/*.report.md >/dev/null 2>&1; then
  ls -lt output/ | head
  LATEST=$(ls -t output/*.report.md | head -1)
  echo "=== $LATEST"; cat "$LATEST"
  echo "=== stats"; cat stats.json
else
  echo "no output yet; check logs/errors.log"; tail -20 logs/errors.log 2>/dev/null || true
fi

### 6.4 Other ways to feed it

```bash
wiki-translation-harness --titles articles.txt           # one title per line, lang:Title allowed
wiki-translation-harness --category "fr:Ville du Kosovo"  # a whole category
wiki-translation-harness --file article.wiki              # local wikitext
wiki-translation-harness --directory raw_articles/        # folder of .wiki files
wiki-translation-harness --title "https://sr.wikipedia.org/wiki/Призрен"   # full URL, any wiki
```

Same command again resumes from cache (free). `--force` retranslates.

In [ ]:
%%bash
# Batch from a file
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
printf "fr:%s\nen:%s\n" "$DEMO_FR" "$DEMO_EN" > articles.txt
cat articles.txt
wiki-translation-harness --config config.workshop.yaml --titles articles.txt

### 6.5 Engines

All five take the same flags; only `--provider` (and its key) changes.
Run whichever you have access to.

In [ ]:
%%bash
# OpenRouter (pay per token): https://openrouter.ai/models
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
export OPENROUTER_API_KEY="${OPENROUTER_API_KEY:-sk-or-...}"
wiki-translation-harness --config config.workshop.yaml --title "$DEMO_EN" --force \
  --provider openrouter --model deepseek/deepseek-v3.2

In [ ]:
%%bash
# Local model: start a llama.cpp server first, in another terminal:
#   llama-server -m qwen3-8b-q5_k_m.gguf --port 8080
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
wiki-translation-harness --config config.workshop.yaml --title "$DEMO_EN" --force \
  --provider local --base-url http://127.0.0.1:8080/v1 --model qwen3-8b-q5-k-m

In [ ]:
%%bash
# Experiential gateway: https://github.com/experientiallabs/experiential
# Key from platform.experientiallabs.ai; model = an alias/id available on your account
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
export EXPLABS_API_KEY="${EXPLABS_API_KEY:-xpl_...}"
wiki-translation-harness --config config.workshop.yaml --title "$DEMO_EN" --force \
  --provider experiential --model "<model id on your account>"

In [ ]:
%%bash
# OpenCode Go: runs the `opencode` CLI under its own login (https://github.com/sst/opencode)
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
wiki-translation-harness --config config.workshop.yaml --title "$DEMO_EN" --force \
  --provider opencode_go

### 6.6 Mixing models in one run

| Flag | Used for |
|---|---|
| `--model` | ordinary chunks |
| `--complex-model` (+ `--complex-provider`) | infoboxes, tables, dense references |
| `--review-model` (+ `--review-provider`) | re-reads the assembled article for meaning errors |
| `--fallback-provider` | switch engine if credits run out |

In [ ]:
%%bash
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
wiki-translation-harness --config config.workshop.yaml --title "$DEMO_EN" --force \
  --model claude-haiku-4-5-20251001 \
  --complex-model claude-sonnet-5 \
  --review-model claude-sonnet-5

### How it evolved

- **v0.1**: OpenRouter only, skill loading, cache, Wikidata verification,
  deterministic fixes (citation language, CS1 parameter names, footnote
  dedup), reports, benchmark mode.
- **v0.2**: blind judge-model evaluation.
- **Since**: skill split so the QA checklist loads only on repair calls
  (~29% fewer prompt tokens per ordinary call); Claude Code CLI engine;
  article-level terminology registry so later chunks reuse earlier
  renderings; Experiential and OpenCode Go engines; complex-chunk and
  review-model routing.

None of these changes touched the translation rules. Judgment lives in the
skill, mechanics in the harness.

🔗 **Open now:** [Changelog](https://github.com/arianit/wiki-translate-harness/blob/main/CHANGELOG.md) · [Commit history](https://github.com/arianit/wiki-translate-harness/commits/main/) · [Experiential](https://github.com/experientiallabs/experiential) · [OpenCode](https://github.com/sst/opencode) · [Kosovo HPC & AI Competence Centre](https://hpc.kren-ks.eu/)

### 🗣 Discussion

- Anything the harness fixes in code (CS1 parameter names, citation language) is something the model got wrong often enough to stop trusting it. What else belongs on that list?
- Running on someone else's API: pricing, rate limits and terms change. When is it worth running our own inference, and on what? (e.g. the Kosovo HPC & AI Competence Centre, hpc.kren-ks.eu)

## 7. Choosing models ✅ / 🔧

What matters here, and general leaderboards mostly don't measure:

- cost per article at 1,000 to 10,000 articles
- clean wikitext out, no chatter around it
- following the skill's rules
- **Albanian quality**, which few public benchmarks cover

Current setup: a cheaper model drafts (Haiku 4.5), a stronger one reviews
and repairs (Sonnet 5). Local models (Qwen3 via llama.cpp) as an even
cheaper first draft: 🔧 explored, not proven.

### Benchmark mode

Same article through several models, then a separate judge model scores
them blind (labelled A to D, random order). Uses `config.openrouter.yaml`
because the model ids are OpenRouter ids. Costs roughly a dollar.

In [ ]:
%%bash
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
wiki-translation-harness benchmark --config config.openrouter.yaml --title "$DEMO_EN" \
  --model deepseek/deepseek-v3.2 \
  --model google/gemini-2.5-flash \
  --model mistralai/mistral-large \
  --model qwen/qwen3-235b-a22b \
  --judge-model anthropic/claude-sonnet-4.5

In [ ]:
%%bash
cd "$WORK/wiki-translate-harness"
cat quality/*/comparison.md 2>/dev/null | head -80 || true
cat quality/*/mapping.json 2>/dev/null || echo "(no benchmark results yet)"

Previous result ([issue #1](https://github.com/arianit/wiki-translate-harness/issues/1)):

| Model | Score | Cost | Speed |
|---|---|---|---|
| deepseek/deepseek-v3.2 | 9/10 | $0.062 | 220 tok/s |
| google/gemini-2.5-flash | 9/10 | $0.128 | 622 tok/s |
| mistralai/mistral-large | 8/10 | $0.542 | 128 tok/s |
| qwen/qwen3-235b-a22b | 6/10 | $0.183 | 177 tok/s |

One article, one judge. A starting point, not a verdict.

🔗 **Open now:** [Issue #1](https://github.com/arianit/wiki-translate-harness/issues/1) · [OpenRouter models and prices](https://openrouter.ai/models)

### 🗣 Discussion

- Can an LLM judge Albanian quality reliably? How would we find out? (Compare its scores with native-speaker editors on the same texts.)
- What goes in a fixed test set? A short biography, a heavily cited medical article, a long philosophy article, a battle with foreign names...
- Draft-then-review on every chunk, or escalate to the stronger model only when validation fails?
- Which models have you seen do well in Albanian, for any task?

## 8. Sharing work through the queue ✅

[`wiki-translate-queue`](https://github.com/arianit/wiki-translate-queue):
a text file in git. Several people and machines work through one list
without translating the same article twice. Claims use
discard-and-redo instead of git merges, because git reports false
conflicts on adjacent short lines.

Line format in `totranslate.txt`: URL or title, then nothing (pending),
`CLAIMED <host> <time>`, `DONE` or `FAILED`.

🔗 **Open now:** [totranslate.txt](https://github.com/arianit/wiki-translate-queue/blob/main/totranslate.txt) · [Finished articles](https://github.com/arianit/wiki-translate-queue/tree/main/output) · [Queue history](https://github.com/arianit/wiki-translate-queue/commits/main/)

In [ ]:
%%bash
cd "$WORK"
[ -d wiki-translation-queue ] || git clone -q https://github.com/arianit/wiki-translate-queue wiki-translation-queue
cd wiki-translation-queue && git pull -q --ff-only
PENDING=$(grep -v '^#' totranslate.txt | grep -v '^[[:space:]]*$' | grep -v $'\t')
echo "pending: $(echo "$PENDING" | grep -c .)"
echo "claimed: $(grep -c $'\tCLAIMED' totranslate.txt)"
echo "done:    $(grep -c $'\tDONE' totranslate.txt)"
echo "failed:  $(grep -c $'\tFAILED' totranslate.txt)"
echo "--- next pending"; echo "$PENDING" | head -5

**The next two cells write to the shared queue** (commit and push). They do
nothing unless `ALLOW_QUEUE_WRITES` is `"yes"` in Setup. Without write
access, fork the queue repo and point `--queue-repo-dir` at your fork.

In [ ]:
%%bash
# Add an article to the queue
[ "$ALLOW_QUEUE_WRITES" = "yes" ] || { echo "skipped: ALLOW_QUEUE_WRITES is not yes"; exit 0; }
cd "$WORK/wiki-translation-queue"
echo "https://fr.wikipedia.org/wiki/$DEMO_SLUG" >> totranslate.txt
git commit -qam "Queue fr:$DEMO_FR" && git push -q && tail -3 totranslate.txt

In [ ]:
%%bash
# Drain it: claim, translate, mark DONE/FAILED, push. One article only.
[ "$ALLOW_QUEUE_WRITES" = "yes" ] || { echo "skipped: ALLOW_QUEUE_WRITES is not yes"; exit 0; }
cd "$WORK/wiki-translate-harness" && source .venv/bin/activate
wiki-translation-harness queue --config config.workshop.yaml \
  --queue-repo-dir "$WORK/wiki-translation-queue" --max-articles 1

Or from your own script. Shown, not run: executing it claims a real line in
the shared queue.

```python
import sys
from pathlib import Path

repo_dir = Path("~/code/wiki-translation-queue").expanduser()
sys.path.insert(0, str(repo_dir))
import queue_lib

claimed = queue_lib.claim_next_pending(repo_dir)   # pull, claim, push
if claimed:
    line_no, url = claimed
    ok = translate(url)                             # your own function
    queue_lib.finish_line(repo_dir, line_no, url, "DONE" if ok else "FAILED")
```

### 🗣 Discussion

- Who decides what goes in the queue? Should the sqwiki community see and edit the list?
- Where does human review sit: before an article leaves the queue, or after it's on sqwiki?

## 9. Not built yet 🧭

| Item | Status | Where to start |
|---|---|---|
| French (and other) source rules in the skill | 🔧 step 5 | skill repo |
| Fixed evaluation set, judge checked against human editors | 🧭 | harness issues |
| Stronger model only when needed | 🧭 | harness issues |
| Local model good enough for a first draft | 🔧 | `--provider local` |
| Own inference | 🧭 | compute partners, e.g. [Kosovo HPC & AI Competence Centre](https://hpc.kren-ks.eu/) |
| Fine-tune a model on reviewed translations | 🧭 | needs reviewed pairs first |
| Tool-using agent vs. harness, measured | 🧭 | harness issues |
| Output as a parallel corpus | 🧭 | [albanian-language-tech-plan](https://github.com/arianit/albanian-language-tech-plan) |

### The long road: an Albanian model

1. Every reviewed article is a training pair: draft in, corrected text out.
2. Experiential can turn traffic into a fine-tuned model you own; a
   LoRA/adapter on a small open model is the other route.
3. Compute: rented GPUs, or a national resource like the
   [Kosovo HPC & AI Competence Centre](https://hpc.kren-ks.eu/)
   (EuroHPC EUROCC project, University of Prishtina and KREN).
4. The [Albanian Language Technology Development Plan](https://github.com/arianit/albanian-language-tech-plan)
   calls for 4 million parallel pairs and an Albanian LLM. This pipeline
   produces parallel en/fr to sq text as a side effect.

None of this is built.

🔗 **Open now:** [Albanian language-tech plan](https://github.com/arianit/albanian-language-tech-plan/blob/main/albanian-language-tech-plan.md) · [Kosovo HPC & AI Competence Centre](https://hpc.kren-ks.eu/) · [Experiential](https://github.com/experientiallabs/experiential) · [Wikipedia licensing (CC BY-SA)](https://en.wikipedia.org/wiki/Wikipedia:Copyrights)

### 🗣 Discussion

- Licensing: translations of Wikipedia are CC BY-SA. A corpus built from them carries that licence, not CC0. Does that fit the language-tech plan's goals?
- Some model providers' terms restrict using their outputs to train other models. Does that limit fine-tuning on drafts they produced? Check each provider's terms before building on it.
- Who could host inference long term: a university, KREN, a Wikimedia affiliate?
- Is fine-tuning worth it, or will general models get good enough at Albanian on their own?

## Contribute

Pick one 🧭 line, or anything from today's discussion, and **open an issue
before you leave**. Small is fine: one disputed term, one broken citation,
one "someone should test this".

| Repo | For |
|---|---|
| [enwiki-sqwiki-translation](https://github.com/arianit/enwiki-sqwiki-translation/issues) | terminology, new source languages (like today's French work) |
| [wiki-translate-harness](https://github.com/arianit/wiki-translate-harness/issues) | engines, verification, fixes, local models, benchmarks |
| [wiki-translate-queue](https://github.com/arianit/wiki-translate-queue/issues) | which articles, coordination |
| [albanian-language-tech-plan](https://github.com/arianit/albanian-language-tech-plan/issues) | the bigger picture |

- **Techies:** a new engine, better verification, local models.
- **Linguists:** terminology, transliteration, source-language rules.
- **Wikipedians:** choose articles, review drafts, report what breaks on sqwiki.
- **Anyone with compute or hosting:** say so. Own inference is a standing ask.

Before anything goes live on sq.wikipedia: a person who reads both
languages reviews the full text and its sources.

In [ ]:
%%bash
# Open an issue from the command line (needs the GitHub CLI, logged in)
gh issue create --repo arianit/wiki-translate-harness \
  --title "<short description>" \
  --body "What I tried:
What happened:
What I expected:"